In [18]:
import json
import os
import re
from pathlib import Path
from rapidfuzz import fuzz

In [3]:
CARDS_DIR = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Code\Cards"
CARD_STRINGS = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Downfall\localization\CardStrings.json"
ENG_CARDS   = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Downfall\localization\eng\cards.json"

In [21]:
def load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"Could not load {path}: {e}")
        return {}

card_strings = load_json(CARD_STRINGS)
eng_cards    = load_json(ENG_CARDS)

cards = {}

for cs_path in sorted(Path(CARDS_DIR).rglob("*.cs")):
    parts = cs_path.relative_to(CARDS_DIR).parts
    # parts = ("Gremlins", "Token", "Rush.cs") or ("Gremlins", "Rush.cs")
    class_name = cs_path.stem
    char  = parts[0] if len(parts) >= 2 else ""
    rarity = parts[1] if len(parts) >= 3 else parts[0] if len(parts) == 2 else ""

    cards[class_name] = {
        "char":  char,
        "type":  rarity,
    }

In [22]:
card_strings

{'champ:Strike': {'NAME': 'Strike', 'DESCRIPTION': 'Deal !D! damage.'},
 'champ:Defend': {'NAME': 'Defend', 'DESCRIPTION': 'Gain !B! Block.'},
 'champ:Taunt': {'NAME': 'Taunt',
  'DESCRIPTION': 'Apply !M! Weak and Vulnerable.',
  'UPGRADE_DESCRIPTION': 'Apply !M! Weak and Vulnerable to ALL enemies.'},
 'champ:Execute': {'NAME': 'Execute',
  'DESCRIPTION': 'Deal !D! damage !cool! times.',
  'UPGRADE_DESCRIPTION': 'Deal !D! damage !cool! times.'},
 'champ:StanceDance': {'NAME': 'Stance Dance',
  'DESCRIPTION': 'Choose a Stance to Enter. NL Trigger its *Skill champ:Bonus.',
  'UPGRADE_DESCRIPTION': 'Choose a Stance to Enter. NL Trigger its *Skill champ:Bonus twice.',
  'EXTENDED_DESCRIPTION': ['Enter champ:Berserker.',
   'Enter champ:Defensive.']},
 'champ:StanceDanceCrown': {'NAME': 'Inspiration',
  'DESCRIPTION': "Retain. NL Enter a Stance you aren't in. NL Exhaust.",
  'UPGRADE_DESCRIPTION': "Retain. NL Enter a Stance you aren't in."},
 'champ:SwordSigil': {'NAME': 'Sigil of Victory',

In [14]:
eng_cards

{'DOWNFALL-FUNCTION_ATTACK_CARD.title': 'function()',
 'DOWNFALL-FUNCTION_ATTACK_CARD.description': '{effects}',
 'DOWNFALL-FUNCTION_SKILL_CARD.title': 'function()',
 'DOWNFALL-FUNCTION_SKILL_CARD.description': '{effects}',
 'DOWNFALL-FUNCTION_POWER_CARD.title': 'function()',
 'DOWNFALL-FUNCTION_POWER_CARD.description': '{effects}',
 'DOWNFALL-DEFEND_AUTOMATON.title': 'Defend',
 'DOWNFALL-DEFEND_AUTOMATON.description': 'Gain {Block:diff()} [gold]Block[/gold].',
 'DOWNFALL-GOTO.description': '',
 'DOWNFALL-GOTO.title': 'Goto',
 'DOWNFALL-REPLICATE.title': 'Replicate',
 'DOWNFALL-REPLICATE.description': 'When [gold]Encoded[/gold], add a copy to your discard pile.',
 'DOWNFALL-STRIKE_AUTOMATON.title': 'Strike',
 'DOWNFALL-STRIKE_AUTOMATON.description': 'Deal {Damage:diff()} damage.',
 'DOWNFALL-BIT_SHIFT.title': 'Bit Shift',
 'DOWNFALL-BIT_SHIFT.description': 'Choose an Encoded card to return to your hand. It gains [gold]Retain[/gold].',
 'DOWNFALL-BRANCH.description': 'Deal {Damage:diff(

In [15]:
cards

{'DefendAutomaton': {'char': 'Automaton', 'type': 'Basic'},
 'Goto': {'char': 'Automaton', 'type': 'Basic'},
 'Replicate': {'char': 'Automaton', 'type': 'Basic'},
 'StrikeAutomaton': {'char': 'Automaton', 'type': 'Basic'},
 'BitShift': {'char': 'Automaton', 'type': 'Common'},
 'Branch': {'char': 'Automaton', 'type': 'Common'},
 'BugBarrage': {'char': 'Automaton', 'type': 'Common'},
 'BuggyMess': {'char': 'Automaton', 'type': 'Common'},
 'Cleanse': {'char': 'Automaton', 'type': 'Common'},
 'CutThrough': {'char': 'Automaton', 'type': 'Common'},
 'DelayedGuard': {'char': 'Automaton', 'type': 'Common'},
 'Deprecate': {'char': 'Automaton', 'type': 'Common'},
 'FineTuning': {'char': 'Automaton', 'type': 'Common'},
 'Frontload': {'char': 'Automaton', 'type': 'Common'},
 'Invalidate': {'char': 'Automaton', 'type': 'Common'},
 'OilSpill': {'char': 'Automaton', 'type': 'Common'},
 'Overheat': {'char': 'Automaton', 'type': 'Common'},
 'PiercingShot': {'char': 'Automaton', 'type': 'Common'},
 'Rob

In [37]:
import json
import re
from pathlib import Path
from rapidfuzz import fuzz
from difflib import SequenceMatcher

# ── paths ─────────────────────────────────────────────────────────────────────
CARDS_DIR    = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Code\Cards"
CARD_STRINGS = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Downfall\localization\CardStrings.json"
ENG_CARDS    = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Downfall\localization\eng\cards.json"
OUTPUT       = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Downfall\localization\eng.json"

THRESHOLD = 90

# ── load ──────────────────────────────────────────────────────────────────────
def load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"Could not load {path}: {e}")
        return {}

old_loc = load_json(CARD_STRINGS)
new_loc = load_json(ENG_CARDS)

print(f"Loaded {len(old_loc)} old loc entries")
print(f"Loaded {len(new_loc)} new loc (base) entries")

# ── scan card classes ─────────────────────────────────────────────────────────
VAR_PATTERN = re.compile(
    r'\b(?:DamageVar|BlockVar|MagicVar|EnergyVar|CounterVar)\s+(\w+)\s*[=;{]'
)

cards_idx = {}
for cs_path in sorted(Path(CARDS_DIR).rglob("*.cs")):
    parts      = cs_path.relative_to(CARDS_DIR).parts
    class_name = cs_path.stem
    char       = parts[0] if len(parts) >= 2 else ""
    rarity     = parts[1] if len(parts) >= 3 else parts[0] if len(parts) == 2 else ""
    source     = cs_path.read_text(encoding="utf-8", errors="ignore")
    cards_idx[class_name] = {
        "char":         char,
        "type":         rarity,
        "placeholders": VAR_PATTERN.findall(source),
    }

print(f"Indexed {len(cards_idx)} card classes")

# ── helpers ───────────────────────────────────────────────────────────────────
def camel_to_snake(name: str) -> str:
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)
    s = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', s)
    return s.upper()

def camel_to_words(name: str) -> str:
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1 \2', name)
    s = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', s)
    return s.strip()

def strip_key(key: str) -> str:
    key = re.sub(r'^[a-zA-Z]+:',  '', key)
    key = re.sub(r'^DOWNFALL-',   '', key)
    key = re.sub(r'\.(title|description|upgrade_description|extended_description)$', '', key, flags=re.I)
    return key.lower().replace('_', '').replace('-', '').replace(' ', '')

def get_old_prefix(key: str) -> str:
    m = re.match(r'^([a-zA-Z]+):', key)
    return m.group(1).lower() if m else ""

LETTER_ORDER = {'D': 0, 'B': 1, 'M': 2, 'E': 3}

def convert_desc(text: str, placeholders: list) -> str:
    text = re.sub(r'\s*NL\s*', '\n', text)
    text = re.sub(r'\*(\w[\w ]*)',    r'[gold]\1[/gold]', text)
    text = re.sub(r'\w+:(\w[\w ]*)', r'[gold]\1[/gold]', text)

    def replace(m):
        letter = m.group(1)
        idx = LETTER_ORDER.get(letter)
        if idx is not None and idx < len(placeholders):
            return f'{{{placeholders[idx]}:diff()}}'
        for p in placeholders:
            if p.lower() == letter.lower():
                return f'{{{p}:diff()}}'
        return ''

    return re.sub(r'!(\w+)!', replace, text)

def merge_upgrade(base: str, upgraded: str) -> str:
    if base == upgraded:
        return base

    def tokenize(s):
        return re.findall(r'\S+|\s+', s)

    matcher = SequenceMatcher(None, tokenize(base), tokenize(upgraded), autojunk=False)
    result_tokens = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        bt = tokenize(base)
        ut = tokenize(upgraded)
        if tag == 'equal':
            result_tokens.extend(bt[i1:i2])
        elif tag == 'replace':
            result_tokens.append(f'{{"IfUpgraded:show:{"".join(ut[j1:j2])}|{"".join(bt[i1:i2])}}}')
        elif tag == 'insert':
            result_tokens.append(f'{{"IfUpgraded:show:{"".join(ut[j1:j2])}|}}')
        elif tag == 'delete':
            result_tokens.append(f'{{"IfUpgraded:show:|{"".join(bt[i1:i2])}}}')

    return ''.join(result_tokens)

# ── index new_loc — collapse upgrade_description into description ─────────────
new_collapsed = {}   # base_key -> {title, description}
for full_key in new_loc:
    base = re.sub(r'\.(title|description|upgrade_description)$', '', full_key, flags=re.I)
    new_collapsed.setdefault(base, {})
    if full_key.endswith('.title'):
        new_collapsed[base]['title'] = new_loc[full_key]
    elif full_key.endswith('.upgrade_description'):
        new_collapsed[base]['upgrade'] = new_loc[full_key]
    elif full_key.endswith('.description'):
        new_collapsed[base]['description'] = new_loc[full_key]

# merge upgrade into description where both exist
new_merged = {}   # stripped_base -> {title, description}
for base, data in new_collapsed.items():
    desc    = data.get('description', '')
    upgrade = data.get('upgrade')
    new_merged[strip_key(base)] = {
        'title':       data.get('title', ''),
        'description': merge_upgrade(desc, upgrade) if upgrade and upgrade != desc else desc,
    }

# ── index old_loc by prefix → {stripped: (key, data)} ───────────────────────
old_stripped   = {strip_key(k): (k, v) for k, v in old_loc.items()}
old_by_prefix  = {}
for old_key, old_data in old_loc.items():
    prefix = get_old_prefix(old_key)
    s      = strip_key(old_key)
    old_by_prefix.setdefault(prefix, {})[s] = (old_key, old_data)

# ── build result — exactly one title + description per CS file ────────────────
result = {}

print("\n── CS files → loc entries ───────────────────────────────────────────────")
print(f"  {'ST':<4}  {'CLASS':<40}  {'METHOD':<24}  CLOSEST")
print(f"  {'-'*4}  {'-'*40}  {'-'*24}  {'-'*60}")

for class_name, info in sorted(cards_idx.items()):
    base_new     = f"DOWNFALL-{camel_to_snake(class_name)}"
    placeholders = info["placeholders"]
    s            = class_name.lower().replace('_', '')
    char_prefix  = info["char"].lower()

    title       = None
    description = None
    method      = "none"
    closest     = ""

    # 1. check new_loc first (it wins)
    if s in new_merged:
        d           = new_merged[s]
        title       = d['title']
        description = d['description']
        method      = "new-exact"
    else:
        best_score_new, best_s_new = max(
            ((fuzz.ratio(s, ns), ns) for ns in new_merged),
            key=lambda x: x[0], default=(0, None)
        )
        if best_s_new and best_score_new >= THRESHOLD:
            d           = new_merged[best_s_new]
            title       = d['title']
            description = d['description']
            method      = f"new-fuzzy({best_score_new})"
            closest     = best_s_new

    # 2. fall back to old_loc
    if title is None:
        candidates = old_by_prefix.get(char_prefix, {}) or old_stripped

        old_data = None
        if s in candidates:
            _, old_data = candidates[s]
            method  = "old-exact-prefix"
            closest = _
        else:
            best_score, best_s = max(
                ((fuzz.ratio(s, os), os) for os in candidates),
                key=lambda x: x[0], default=(0, None)
            )
            if best_s:
                closest = f"{candidates[best_s][0]}  (score={best_score})"
            if best_s and best_score >= THRESHOLD:
                _, old_data = candidates[best_s]
                method = f"old-fuzzy-prefix({best_score})"
            else:
                best_score2, best_s2 = max(
                    ((fuzz.ratio(s, os), os) for os in old_stripped),
                    key=lambda x: x[0], default=(0, None)
                )
                if best_s2:
                    closest = f"{old_stripped[best_s2][0]}  (score={best_score2})"
                if best_s2 and best_score2 >= THRESHOLD:
                    _, old_data = old_stripped[best_s2]
                    method = f"old-fuzzy-any({best_score2})"

        if old_data:
            base_desc    = convert_desc(old_data.get("DESCRIPTION", ""), placeholders)
            upgrade_desc = convert_desc(old_data.get("UPGRADE_DESCRIPTION", ""), placeholders) \
                           if "UPGRADE_DESCRIPTION" in old_data else None
            title       = old_data.get("NAME") or camel_to_words(class_name)
            description = merge_upgrade(base_desc, upgrade_desc) \
                          if upgrade_desc and upgrade_desc != base_desc else base_desc

    # 3. nothing found — use class name as title, empty description
    if title is None:
        title       = camel_to_words(class_name)
        description = ""
        method      = "fallback"

    result[f"{base_new}.title"]       = title
    result[f"{base_new}.description"] = description

    status = "BASE" if f"{base_new}.title" in new_loc else "NEW "
    print(f"  {status}  {class_name:<40}  {method:<24}  {closest}")

# ── sort & save ───────────────────────────────────────────────────────────────
result = dict(sorted(result.items()))
with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"\n✓ {len(result)} entries written to {OUTPUT}")
print(f"  Expected: {len(cards_idx) * 2} keys ({len(cards_idx)} cards × 2)")
print(f"  Actual:   {len(result)} keys")

Loaded 735 old loc entries
Loaded 421 new loc (base) entries
Indexed 733 card classes

── CS files → loc entries ───────────────────────────────────────────────
  ST    CLASS                                     METHOD                    CLOSEST
  ----  ----------------------------------------  ------------------------  ------------------------------------------------------------
  NEW   AdrenalArmor                              old-exact-prefix          champ:AdrenalArmor
  NEW   AdvancingGuard                            old-exact-prefix          hexamod:AdvancingGuard
  NEW   AggressiveDefense                         old-exact-prefix          Gremlin:AggressiveDefense
  NEW   AllOut                                    fallback                  champ:CalledShot  (score=62.5)
  BASE  Allocate                                  new-exact                 
  BASE  Altar                                     new-exact                 
  NEW   Amass                                     old-exact-p

In [90]:
# %% [1] Imports & Paths
import json
import re
from pathlib import Path
from rapidfuzz import fuzz
from difflib import SequenceMatcher

CARDS_DIR    = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Code\Cards"
CARD_STRINGS = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Downfall\localization\CardStrings.json"
ENG_CARDS    = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Downfall\localization\eng\cards.json"
OUTPUT       = r"C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Downfall\localization\eng.json"
THRESHOLD    = 90

In [91]:
# %% [2] Load JSON files
def load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"Could not load {path}: {e}")
        return {}

old_loc = load_json(CARD_STRINGS)
new_loc = load_json(ENG_CARDS)
print(f"old_loc : {len(old_loc)} entries")
print(f"new_loc : {len(new_loc)} entries")

old_loc : 735 entries
new_loc : 421 entries


In [93]:
# %% [3] Scan CS card classes
VAR_PATTERN = re.compile(
    r'\b(?:DamageVar|BlockVar|MagicVar|EnergyVar|CounterVar)\s+(\w+)\s*[=;{]'
)

cards_idx = {}
for cs_path in sorted(Path(CARDS_DIR).rglob("*.cs")):
    parts      = cs_path.relative_to(CARDS_DIR).parts
    class_name = cs_path.stem
    char       = parts[0] if len(parts) >= 2 else ""
    rarity     = parts[1] if len(parts) >= 3 else parts[0] if len(parts) == 2 else ""
    source     = cs_path.read_text(encoding="utf-8", errors="ignore")
    cards_idx[class_name] = {
        "char":         char,
        "type":         rarity,
        "placeholders": VAR_PATTERN.findall(source),
    }

print(f"Indexed {len(cards_idx)} card classes")
for name, info in list(cards_idx.items())[:5]:
    print(f"  {name:30s}  char={info['char']:15s}  vars={info['placeholders']}")

Indexed 733 card classes
  DefendAutomaton                 char=Automaton        vars=[]
  Goto                            char=Automaton        vars=[]
  Replicate                       char=Automaton        vars=[]
  StrikeAutomaton                 char=Automaton        vars=[]
  BitShift                        char=Automaton        vars=[]


In [102]:
# %% [4] Helper functions
def camel_to_snake(name: str) -> str:
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)
    s = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', s)
    return s.upper()

def camel_to_words(name: str) -> str:
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1 \2', name)
    s = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', s)
    return s.strip()

def strip_key(key: str) -> str:
    key = re.sub(r'^[a-zA-Z]+:',  '', key)
    key = re.sub(r'^DOWNFALL-',   '', key)
    key = re.sub(r'\.(title|description|upgrade_description|extended_description)$', '', key, flags=re.I)
    return key.lower().replace('_', '').replace('-', '').replace(' ', '')

def get_old_prefix(key: str) -> str:
    m = re.match(r'^([a-zA-Z]+):', key)
    return m.group(1).lower() if m else ""

LETTER_ORDER    = {'D': 0, 'B': 1, 'E': 2}
LETTER_FALLBACK = {'D': 'Damage', 'B': 'Block', 'E': 'Energy'}

def replace_energy(text: str) -> str:
    """Replace [E][E][E]... sequences with energyIcons(n)"""
    def repl(m):
        count = m.group(0).count('[E]')
        return f'{{energyPrefix:energyIcons({count})}}'
    return re.sub(r'(\[E\])+', repl, text)

def convert_desc(text: str, placeholders: list, class_name: str = "") -> str:
    text = re.sub(r'\s*NL\s*', '\n', text)
    text = replace_energy(text)   # must be before other replacements
    text = re.sub(r'\*([A-Z][a-zA-Z ]*?)(?=[.,!?\n]|$)', r'[gold]\1[/gold]', text)
    text = re.sub(r'[a-z]+:([A-Z][a-zA-Z ]*)', r'[gold]\1[/gold]', text)

    magic_name     = f"{class_name}Power" if class_name else "Magic"
    non_magic_vars = [p for p in placeholders if p.lower() != 'magic']

    def replace(m):
        letter = m.group(1)
        if letter.upper() == 'M':
            return f'{{{magic_name}:diff()}}'
        for p in non_magic_vars:
            if p.lower() == letter.lower():
                return f'{{{p}:diff()}}'
        idx = LETTER_ORDER.get(letter.upper())
        if idx is not None and idx < len(non_magic_vars):
            return f'{{{non_magic_vars[idx]}:diff()}}'
        if letter.upper() in LETTER_FALLBACK:
            return f'{{{LETTER_FALLBACK[letter.upper()]}:diff()}}'
        return f'{{{letter}:diff()}}'

    return re.sub(r'!(\w+)!', replace, text)

ENERGY_ICON_RE = re.compile(r'\{energyPrefix:energyIcons\(\d+\)\}')

def merge_upgrade(base: str, upgraded: str) -> str:
    if not upgraded or base == upgraded:
        return base

    def tokenize(s): return re.findall(r'\S+|\s+', s)
    bt = tokenize(base)
    ut = tokenize(upgraded)
    matcher = SequenceMatcher(None, bt, ut, autojunk=False)
    out = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            out.extend(bt[i1:i2])
        elif tag == 'replace':
            base_chunk = ''.join(bt[i1:i2])
            upgr_chunk = ''.join(ut[j1:j2])
            # if both sides are energy icons but different counts → dynamic form
            if ENERGY_ICON_RE.fullmatch(base_chunk.strip()) and \
               ENERGY_ICON_RE.fullmatch(upgr_chunk.strip()):
                out.append('{Energy:energyIcons()}')
            else:
                out.append(f'{{IfUpgraded:show:{upgr_chunk}|{base_chunk}}}')
        elif tag == 'insert':
            out.append(f'{{IfUpgraded:show:{"".join(ut[j1:j2])}|}}')
        elif tag == 'delete':
            out.append(f'{{IfUpgraded:show:|{"".join(bt[i1:i2])}}}')

    return ''.join(out)

# sanity checks
print(replace_energy("Costs [E] less."))               # {energyPrefix:energyIcons(1)}
print(replace_energy("Pay [E][E] to play."))            # {energyPrefix:energyIcons(2)}
print(merge_upgrade(
    "{energyPrefix:energyIcons(2)} Cost.",
    "{energyPrefix:energyIcons(1)} Cost."
))                                                       # {Energy:energyIcons()} Cost.
print(convert_desc("Deal !D! damage. NL Gain !B! Block. NL Gain !M! Strength.", [], "YouAreMine"))
print(convert_desc("Deal !D! damage. NL Gain !B! Block.", ["Damage", "Block"], "TwinStrike"))
print(convert_desc("Apply !M! Weak.", ["Magic"], "YouAreMine"))
print(merge_upgrade("Deal 5 damage.", "Deal 8 damage."))

Costs {energyPrefix:energyIcons(1)} less.
Pay {energyPrefix:energyIcons(2)} to play.
{Energy:energyIcons()} Cost.
Deal {Damage:diff()} damage.
Gain {Block:diff()} Block.
Gain {YouAreMinePower:diff()} Strength.
Deal {Damage:diff()} damage.
Gain {Block:diff()} Block.
Apply {YouAreMinePower:diff()} Weak.
Deal {IfUpgraded:show:8|5} damage.


In [103]:
# %% [5] Index new_loc — collapse upgrade into description
new_merged    = {}
_new_collapsed = {}

for full_key in new_loc:
    base = re.sub(r'\.(title|description|upgrade_description)$', '', full_key, flags=re.I)
    _new_collapsed.setdefault(base, {})
    if full_key.endswith('.title'):
        _new_collapsed[base]['title'] = new_loc[full_key]
    elif full_key.endswith('.upgrade_description'):
        _new_collapsed[base]['upgrade'] = new_loc[full_key]
    elif full_key.endswith('.description'):
        _new_collapsed[base]['description'] = new_loc[full_key]

for base, data in _new_collapsed.items():
    desc    = data.get('description', '')
    upgrade = data.get('upgrade')
    new_merged[strip_key(base)] = {
        'title':       data.get('title', ''),
        'description': merge_upgrade(desc, upgrade) if upgrade and upgrade != desc else desc,
    }

print(f"new_merged: {len(new_merged)} base entries")
for k, v in list(new_merged.items())[:10]:
    print(f"  {k!r}: title={v['title']!r}  desc={v['description'][:60]!r}")

new_merged: 218 base entries
  'functionattackcard': title='function()'  desc='{effects}'
  'functionskillcard': title='function()'  desc='{effects}'
  'functionpowercard': title='function()'  desc='{effects}'
  'defendautomaton': title='Defend'  desc='Gain {Block:diff()} [gold]Block[/gold].'
  'goto': title='Goto'  desc=''
  'replicate': title='Replicate'  desc='When [gold]Encoded[/gold], add a copy to your discard pile.'
  'strikeautomaton': title='Strike'  desc='Deal {Damage:diff()} damage.'
  'bitshift': title='Bit Shift'  desc='Choose an Encoded card to return to your hand. It gains [gol'
  'branch': title='Branch'  desc='Deal {Damage:diff()} damage or gain {Block:diff()} [gold]Blo'
  'branchblock': title='Branch: Block'  desc=''


In [104]:
# %% [6] Index old_loc by prefix
old_stripped  = {strip_key(k): (k, v) for k, v in old_loc.items()}
old_by_prefix = {}
for old_key, old_data in old_loc.items():
    prefix = get_old_prefix(old_key)
    s      = strip_key(old_key)
    old_by_prefix.setdefault(prefix, {})[s] = (old_key, old_data)

print(f"old_stripped : {len(old_stripped)} entries")
print(f"Prefixes     : {sorted(old_by_prefix.keys())}")

old_stripped : 724 entries
Prefixes     : ['', 'champ', 'collector', 'gremlin', 'guardian', 'hexamod', 'slimebound', 'sneckomod']


In [112]:
# %% [7] Build result — exactly one title + description per CS file
result  = {}
entries = []

for class_name, info in sorted(cards_idx.items()):
    base_new     = f"DOWNFALL-{camel_to_snake(class_name)}"
    placeholders = info["placeholders"]
    s            = class_name.lower().replace('_', '')
    char_prefix  = info["char"].lower()

    title        = None
    description  = None
    closest_key  = ""
    old_desc_raw = ""
    status       = 0

    # 1. new_loc wins
    if s in new_merged:
        title       = new_merged[s]['title']
        description = new_merged[s]['description']
        closest_key = "Existing"
        status      = 2
    else:
        best_score_new, best_s_new = max(
            ((fuzz.ratio(s, ns), ns) for ns in new_merged),
            key=lambda x: x[0], default=(0, None)
        )
        if best_s_new and best_score_new >= THRESHOLD:
            title       = new_merged[best_s_new]['title']
            description = new_merged[best_s_new]['description']
            closest_key = f"{best_s_new} {best_score_new}%"
            status      = 2

    # 2. old_loc fallback
    if title is None:
        candidates = old_by_prefix.get(char_prefix) or old_stripped
        old_data   = None
        best_miss  = ""   # track best even if under threshold

        if s in candidates:
            old_key, old_data = candidates[s]
            closest_key       = strip_key(old_key)
        else:
            best_score, best_s = max(
                ((fuzz.ratio(s, os), os) for os in candidates),
                key=lambda x: x[0], default=(0, None)
            )
            if best_s:
                best_miss = f"{strip_key(candidates[best_s][0])} {best_score}%"
            if best_s and best_score >= THRESHOLD:
                old_key, old_data = candidates[best_s]
                closest_key       = best_miss
            else:
                best_score2, best_s2 = max(
                    ((fuzz.ratio(s, os), os) for os in old_stripped),
                    key=lambda x: x[0], default=(0, None)
                )
                if best_s2:
                    best_miss = f"{strip_key(old_stripped[best_s2][0])} {best_score2}%"
                if best_s2 and best_score2 >= THRESHOLD:
                    old_key, old_data = old_stripped[best_s2]
                    closest_key       = best_miss
                else:
                    closest_key = f"⚠ closest: {best_miss}"

        if old_data:
            old_desc_raw = old_data.get("DESCRIPTION", "")
            base_desc    = convert_desc(old_desc_raw, placeholders, class_name)
            upgrade_desc = convert_desc(old_data.get("UPGRADE_DESCRIPTION", ""), placeholders, class_name) \
                           if "UPGRADE_DESCRIPTION" in old_data else None
            title        = old_data.get("NAME") or camel_to_words(class_name)
            description  = merge_upgrade(base_desc, upgrade_desc) \
                           if upgrade_desc and upgrade_desc != base_desc else base_desc
            status       = 1

    # 3. fallback
    if title is None:
        title       = camel_to_words(class_name)
        description = ""
        status      = 0
        # closest_key already set to "⚠ closest: ..." above

    result[f"{base_new}.title"]       = title
    result[f"{base_new}.description"] = description
    entries.append((status, class_name, closest_key, old_desc_raw, description))

# ── print sorted: not found, old found, existing ─────────────────────────────
STATUS_LABEL = {0: "⚠ MISS", 1: "✚ OLD", 2: "✓ BASE"}

print(f"{'CLASS':<25} {'STATUS':<7} {'CLOSEST':<35} {'OLD DESC':<50} NEW DESC")
print("─" * 150)

for status in [0, 1, 2]:
    group = [e for e in entries if e[0] == status]
    if not group:
        continue
    print(f"\n── {STATUS_LABEL[status]} ({len(group)}) ──")
    for _, class_name, closest_key, old_desc_raw, description in group:
        old_p = old_desc_raw[:48].replace('\n', '↵') + ("…" if len(old_desc_raw) > 48 else "")
        new_p = description[:48].replace('\n', '↵')  + ("…" if len(description)  > 48 else "")
        print(f"  {class_name:<15} {STATUS_LABEL[status]:<7} {closest_key:<15} {old_p:<50} {new_p}")

print("\n" + "─" * 150)
print(f"  ⚠ Not found : {sum(1 for e in entries if e[0]==0)}")
print(f"  ✚ From old  : {sum(1 for e in entries if e[0]==1)}")
print(f"  ✓ Existing  : {sum(1 for e in entries if e[0]==2)}")
print(f"  Total cards : {len(cards_idx)}  keys: {len(result)}  expected: {len(cards_idx)*2}")

CLASS                     STATUS  CLOSEST                             OLD DESC                                           NEW DESC
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

── ⚠ MISS (108) ──
  AllOut          ⚠ MISS  ⚠ closest: calledshot 62.5%                                                    
  Amber           ⚠ MISS  ⚠ closest: ember 80.0%                                                    
  Amethyst        ⚠ MISS  ⚠ closest: armstheft 58.82352941176471%                                                    
  AncientPower    ⚠ MISS  ⚠ closest: ancientconstruct 64.28571428571428%                                                    
  Aquamarine      ⚠ MISS  ⚠ closest: youaremine 60.0%                                                    
  AwakenedCardModel ⚠ MISS  ⚠ closest: markedcard 59.25925925925925%                                                    
  BattlePlan      ⚠ MI

In [108]:
# %% [8] Sort & save
result = dict(sorted(result.items()))
with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"✓ Written to {OUTPUT}")
print(f"  {len(result)} keys  ({len(result)//2} cards)")
for k, v in list(result.items())[:6]:
    print(f"  {k!r}: {v!r}")

✓ Written to C:\Users\lamal\Desktop\Programmieren\sts2mods\Downfall\Downfall\localization\eng.json
  1466 keys  (733 cards)
  'DOWNFALL-ADRENAL_ARMOR.description': 'Gain {Block:diff()} Block.\nGain {AdrenalArmorPower:diff()} temporary Strength.'
  'DOWNFALL-ADRENAL_ARMOR.title': 'Adrenal Armor'
  'DOWNFALL-ADVANCING_GUARD.description': 'Gain {Block:diff()} Block.\n[gold]Advance[/gold].'
  'DOWNFALL-ADVANCING_GUARD.title': 'Advancing Guard'
  'DOWNFALL-AGGRESSIVE_DEFENSE.description': 'Deal {Damage:diff()} damage.\nWhenever you gain Block this turn, the enemy loses {AggressiveDefensePower:diff()} HP.'
  'DOWNFALL-AGGRESSIVE_DEFENSE.title': 'Aggressive Defense'
